# 07. 난이도 라우팅 — frontier 토큰 비용 절감

Sonnet 같은 frontier 모델은 강력하지만 토큰 비용($3/$15 per MTok)이 큽니다. 그런데 고객 문의의 상당수는 **운영시간·분실신고처럼 작은 모델로 충분한 질문**입니다. 이 노트북에서는 decision layer를 **traffic router**로 사용합니다: 질문의 난이도를 `/v1/systemone`으로 판정해서, **simple은 Haiku 4.5($1/$5)로, complex는 Sonnet 4.5($3/$15)로** 보냅니다. 생성은 양쪽 모두 Bedrock — 라우터만 self-hosted decision model입니다.

notebook 05(Bedrock Intelligent Prompt Routing)와의 관계:

| 비교 | Bedrock IPR (notebook 05) | 난이도 라우팅 (이 노트북) |
|---|---|---|
| 선택 대상 | 같은 family의 Bedrock 모델 (Lite/Pro) | **비용 등급이 다른 두 모델 (Haiku vs Sonnet)** — family 제약 없음 |
| 라우터 | 관리형 (내장 예측 모델) | 직접 구성 (decision layer) |
| 라우터 비용 | $1 / 1,000건 | 수십~수백 ms, 한계비용 ~0 |
| 제어 | responseQualityDifference 1개 | rubric·선택지·폴백을 직접 설계 |

두 방식은 보완재입니다 — 관리형으로 충분하면 IPR로, rubric과 폴백 방향까지 직접 설계하려면 decision layer로.


In [1]:
import sys, urllib.request; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common 경로 (repo 내 상대 위치)
import ws_common as W
print("decision server:", f"http://{W.STRANDS_HOST}:{W.STRANDS_PORT}")
print("health:", urllib.request.urlopen(f"http://{W.STRANDS_HOST}:{W.STRANDS_PORT}/health", timeout=5).status)
client, sonnet = W.make_clients()
haiku  = W.BedrockLLM(client, W.BEDROCK_MODEL_IDS["haiku-4-5"], W.PRICING["haiku-4-5"])
router = W.SystemOneDecision()

decision server: http://localhost:8004
health: 200


## 1. 라우터 = 2-choice decision

난이도 rubric(`W.ROUTER_QUESTION`)과 선택지 `["complex", "simple"]`로 `/v1/systemone`을 호출합니다.

> **폴백 안전장치**: 선택지 순서가 중요합니다. 라우터 출력이 유효하지 않으면 decision backend는 `choices[0]`으로 폴백하는데, `complex`를 첫 번째에 두었으므로 **오류 시 항상 Sonnet(고품질 쪽)으로** 갑니다. 비용 최적화가 품질 사고로 이어지지 않게 하는 설계입니다.


In [2]:
print("rubric:", W.ROUTER_QUESTION, "\n")
for q in (W.routing_questions()[0]["q"], W.routing_questions()[5]["q"]):
    d = router.decide(state_text=q, question=W.ROUTER_QUESTION,
                      choices=W.ROUTER_CHOICES, decision_id="route")
    print(f"[{d.label:<7s}] conf {d.confidence:.3f} {d.latency_ms:>6,.0f}ms | {q[:60]}")

rubric: 다음 은행 고객 문의의 난이도를 분류하라. simple = 한두 문장으로 답할 수 있는 단일 사실·절차 안내(운영시간, 메뉴 위치, 단순 신고/변경 방법 등). complex = 여러 단계의 추론이나 여러 항목의 비교, 규정·세금·수수료 해석, 고객 상황에 따른 판단이 필요한 문의. 

[simple ] conf 0.963    155ms | 고객센터 운영 시간이 어떻게 되나요?


[complex] conf 0.738    153ms | 급여계좌를 A은행에서 B은행으로 옮기면서 자동이체 7건과 카드 결제계좌를 함께 이전하려고 합니다. 연체나 불


## 2. 실험 문항 — 난이도 혼합 10문항

단순 FAQ 5문항(simple)과 여러 단계 추론·규정 해석이 필요한 5문항(complex)입니다. `gold`는 출제 의도입니다.


In [3]:
from IPython.display import Markdown, display
rows = ["| # | 질문 | 출제 의도 |", "|---|---|---|"]
for i, item in enumerate(W.routing_questions(), 1):
    rows.append(f"| {i} | {item['q']} | {item['gold']} |")
display(Markdown("\n".join(rows)))

| # | 질문 | 출제 의도 |
|---|---|---|
| 1 | 고객센터 운영 시간이 어떻게 되나요? | simple |
| 2 | 체크카드 비밀번호는 앱 어디에서 변경할 수 있나요? | simple |
| 3 | 카드를 분실했는데 분실 신고는 어떻게 하나요? | simple |
| 4 | 해외 ATM에서도 출금이 가능한가요? | simple |
| 5 | 계좌 잔액 증명서는 어디서 발급받나요? | simple |
| 6 | 급여계좌를 A은행에서 B은행으로 옮기면서 자동이체 7건과 카드 결제계좌를 함께 이전하려고 합니다. 연체나 불이익 없이 진행하는 순서를 알려주세요. | complex |
| 7 | 해외 송금이 중개은행 수수료로 일부 차감되어 도착했습니다. 수수료 부담 방식(OUR/SHA/BEN)의 차이와 차감분 환불 가능 여부를 설명해 주세요. | complex |
| 8 | USD 5,000 환전 기준으로 환율 우대 80%와 송금 수수료 면제 중 어느 쪽이 유리한지 계산 과정을 포함해 비교해 주세요. | complex |
| 9 | 연 매출 3억원 개인사업자입니다. 사업자 대출과 개인 신용대출을 함께 이용할 때 DSR 규제가 각각 어떻게 적용되는지 설명해 주세요. | complex |
| 10 | 미성년 자녀 명의 계좌로 매달 50만원씩 이체해 주려고 합니다. 증여세 신고 기준 금액과 신고 절차를 알려주세요. | complex |

## 3. Baseline — 10문항 전부 Sonnet 4.5

먼저 라우팅 없이 전부 frontier 모델로 보냈을 때의 토큰 비용을 측정합니다. `BedrockLLM`은 호출마다 실제 토큰 수로 비용(`cost_usd`)을 계산합니다.


In [4]:
baseline = []
for item in W.routing_questions():
    r = sonnet.generate(system=W.GEN_SYSTEM, user=item["q"], max_tokens=300)
    baseline.append({"backend": "sonnet", "cost_usd": r.cost_usd,
                     "input_tokens": r.usage["input_tokens"],
                     "output_tokens": r.usage["output_tokens"],
                     "latency_ms": r.latency_ms})
    print(f"${r.cost_usd:.5f} {r.latency_ms:>7,.0f}ms | {item['q'][:55]}")
print(f'\n합계: ${sum(b["cost_usd"] for b in baseline):.4f}')

$0.00455   4,015ms | 고객센터 운영 시간이 어떻게 되나요?


$0.00485   3,770ms | 체크카드 비밀번호는 앱 어디에서 변경할 수 있나요?


$0.00484   4,211ms | 카드를 분실했는데 분실 신고는 어떻게 하나요?


$0.00481   3,889ms | 해외 ATM에서도 출금이 가능한가요?


$0.00483   3,756ms | 계좌 잔액 증명서는 어디서 발급받나요?


$0.00501   4,587ms | 급여계좌를 A은행에서 B은행으로 옮기면서 자동이체 7건과 카드 결제계좌를 함께 이전하려고 합니다. 


$0.00500   4,384ms | 해외 송금이 중개은행 수수료로 일부 차감되어 도착했습니다. 수수료 부담 방식(OUR/SHA/BEN)


$0.00496   4,196ms | USD 5,000 환전 기준으로 환율 우대 80%와 송금 수수료 면제 중 어느 쪽이 유리한지 계산 


$0.00498   4,361ms | 연 매출 3억원 개인사업자입니다. 사업자 대출과 개인 신용대출을 함께 이용할 때 DSR 규제가 각각


$0.00495   4,397ms | 미성년 자녀 명의 계좌로 매달 50만원씩 이체해 주려고 합니다. 증여세 신고 기준 금액과 신고 절차

합계: $0.0488


## 4. Routed — simple은 Haiku, complex는 Sonnet

같은 10문항을 라우터를 거쳐 처리합니다. Haiku도 Bedrock 토큰 과금이므로 **모든 호출의 실제 토큰 비용**을 그대로 집계합니다 — 절감은 전적으로 "같은 질문을 더 싼 모델이 답했는가"에서 나옵니다.


In [5]:
routed, correct = [], 0
for item in W.routing_questions():
    d = router.decide(state_text=item["q"], question=W.ROUTER_QUESTION,
                      choices=W.ROUTER_CHOICES, decision_id="route")
    correct += (d.label == item["gold"])
    target, name = (haiku, "haiku") if d.label == "simple" else (sonnet, "sonnet")
    r = target.generate(system=W.GEN_SYSTEM, user=item["q"], max_tokens=300)
    routed.append({"backend": name, "cost_usd": r.cost_usd,
                   "input_tokens": r.usage["input_tokens"],
                   "output_tokens": r.usage["output_tokens"],
                   "latency_ms": r.latency_ms, "router_ms": d.latency_ms})
    print(f"route={d.label:<7s} (conf {d.confidence:.2f}, {d.latency_ms:>5,.0f}ms)"
          f" -> {name:<6s} ${r.cost_usd:.5f} | {item['q'][:45]}")
print(f'\n라우터 정확도(출제 의도 기준): {correct}/10')

route=simple  (conf 0.96,   109ms) -> haiku  $0.00139 | 고객센터 운영 시간이 어떻게 되나요?


route=simple  (conf 0.89,   153ms) -> haiku  $0.00162 | 체크카드 비밀번호는 앱 어디에서 변경할 수 있나요?


route=simple  (conf 0.84,   153ms) -> haiku  $0.00161 | 카드를 분실했는데 분실 신고는 어떻게 하나요?


route=simple  (conf 0.76,   151ms) -> haiku  $0.00161 | 해외 ATM에서도 출금이 가능한가요?


route=simple  (conf 0.84,   154ms) -> haiku  $0.00161 | 계좌 잔액 증명서는 어디서 발급받나요?


route=complex (conf 0.74,   109ms) -> sonnet $0.00501 | 급여계좌를 A은행에서 B은행으로 옮기면서 자동이체 7건과 카드 결제계좌를 함께 이


route=complex (conf 0.91,   155ms) -> sonnet $0.00500 | 해외 송금이 중개은행 수수료로 일부 차감되어 도착했습니다. 수수료 부담 방식(OU


route=complex (conf 0.92,   154ms) -> sonnet $0.00496 | USD 5,000 환전 기준으로 환율 우대 80%와 송금 수수료 면제 중 어느 쪽


route=complex (conf 0.94,   156ms) -> sonnet $0.00498 | 연 매출 3억원 개인사업자입니다. 사업자 대출과 개인 신용대출을 함께 이용할 때 


route=complex (conf 0.81,   153ms) -> sonnet $0.00495 | 미성년 자녀 명의 계좌로 매달 50만원씩 이체해 주려고 합니다. 증여세 신고 기준

라우터 정확도(출제 의도 기준): 10/10


## 5. 비용 비교 — 솔직한 계산

토큰 비용 절감률과 월 환산액을 봅니다. 라우팅 판단 자체는 decision layer가 이미 쓰고 있는 GPU에서 수행되므로 **라우터의 추가 비용은 0**입니다 — 만약 라우팅만을 위해 GPU를 새로 띄운다면 그 고정비와 절감액을 비교해야 합니다 (아래 표의 설명 참조).


In [6]:
report = W.routing_cost_report(baseline, routed, req_per_day=10_000)

| 항목 | 전부 Sonnet (baseline) | 난이도 라우팅 |
|---|---:|---:|
| 질문 수 | 10 | 10 |
| Sonnet / Haiku 호출 수 | 10 / 0 | 5 / 5 |
| Bedrock 토큰 (입력/출력) | 1,348/2,982 | 1,348/2,956 |
| Bedrock 토큰 비용 (10문항) | $0.0488 | $0.0327 |
| 월 환산 토큰 비용 (하루 10,000건 기준) | $1,463 | $982 |

**토큰 비용 절감률: 32.9%** (하루 10,000건 기준 월 $481 절감)

정직한 비용 계산: 라우팅 판단은 이 아키텍처가 agent의 D1-D8 결정을 위해 이미 운용 중인 decision layer GPU(ml.g5.2xlarge, 약 $1.515/시간, 월 약 $1,091)에서 수행되므로, 라우터 자체의 Bedrock 호출 비용은 0입니다. 라우팅만을 위해 GPU를 새로 띄운다면 위 월 절감액과 GPU 고정비를 비교해 판단해야 합니다.

### 관찰 포인트

- **품질 가드레일**: complex 문의는 전부 Sonnet이 답했고, 라우터 오류 시에도 폴백이 `choices[0]=complex`(Sonnet)입니다. 절감은 "작은 모델로 충분한 질문"에서만 발생합니다.
- **라우터 자체는 공짜에 가깝습니다**: 판정당 수십~수백 ms, Bedrock 호출 없음. IPR의 $1/1,000건과 비교해 보세요.
- **절감의 상한은 가격비**입니다: Haiku는 Sonnet의 1/3 가격이므로, simple 비율이 50%면 산술적으로 최대 ~33% 절감입니다. 실제 수치는 답변 길이(출력 토큰) 분포에 따라 달라집니다 — 위 실측값과 비교해 보세요.
- **rubric 문구에 민감합니다**: notebook 05에서 본 것과 같은 교훈 — simple/complex의 기준 문구를 바꾸면 라우팅 분포가 달라집니다. 운영 전에는 실제 트래픽 샘플로 정확도를 검증해야 합니다.
- 절감률은 **트래픽의 simple 비율**에 비례합니다. 이 실습은 5:5 구성이지만, 실제 고객센터 FAQ 비율(통상 더 높음)로 재계산해 보세요.


### 직접 해보기

- `req_per_day`를 바꿔 월 환산 절감액이 어떻게 움직이는지 확인해 보세요.
- 3-way 라우팅: 선택지를 `["complex", "medium", "simple"]`로 늘리고, simple은 모델 호출 없이 **canned FAQ 응답**으로 처리해 보세요 — 절감률이 어디까지 올라가나요?
- confidence 기반 라우팅: `d.confidence`가 낮으면 simple 판정이어도 Sonnet으로 보내는 보수적 정책을 만들어 보세요 (calibrated confidence라서 가능한 설계입니다).
- `W.scenario(1)["question"]` 같은 상담 시나리오 발화를 라우터에 넣어 어느 쪽으로 분류되는지 확인해 보세요.
